# INE Demographics Data Pipeline

Fetches complete Spanish population demographics from INE API (1971-2022):
- All 54 provinces + National Total
- Ages 0-99 (individual years)
- Gender breakdown (Male, Female, Total)
- Exports in wide format with Newborns column (ages 0-2)

In [8]:
# Setup and imports
import requests
import pandas as pd
from datetime import datetime
from pathlib import Path
import re
from pydantic import BaseModel

class Demographic(BaseModel):
    """Demographics data model"""
    age: int | None
    gender: str
    year: int
    population: int
    province: str
    source_url: str
    fetched_at: datetime

In [9]:
# Fetch data from INE API
def fetch_ine_population_by_age_sex(table_id: str = "31304"):
    """Fetch population data from INE Table 31304"""
    base_url = "https://servicios.ine.es/wstempus/js/EN/DATOS_TABLA"
    url = f"{base_url}/{table_id}?tip=AM"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

In [10]:
# Parse raw data with deduplication
def parse_demographics(raw_data: list) -> list[Demographic]:
    """Parse INE data: extract age, gender, province; deduplicate (July 1st only)"""
    records = []
    
    for series in raw_data:
        name = series.get('Nombre', '')
        parts = [p.strip() for p in name.split('.')]
        
        # Extract age
        age_match = re.search(r'(\d+)\s+years?\s+old', name)
        if age_match:
            age = int(age_match.group(1))
        elif "All ages" in name:
            age = None
        else:
            continue
        
        # Extract gender
        if "Males" in name:
            gender = "Male"
        elif "Females" in name:
            gender = "Female"
        elif "Total" in name:
            gender = "Total"
        else:
            continue
        
        # Extract province (pattern-based)
        if "National Total" in name:
            province = "National Total"
        elif re.match(r'^\d+\s+years?\s+old\.', name) or name.startswith('All ages.'):
            province = parts[1] if len(parts) > 1 else "Unknown"
        else:
            province = "Unknown"
        
        # Deduplication: take only July 1st measurement per year
        year_data = {}
        for datapoint in series.get('Data', []):
            year = datapoint.get('Anyo')
            population = datapoint.get('Valor')
            period = datapoint.get('T3_Periodo', '')
            
            if year and population is not None:
                if year not in year_data or 'July' in period or '1 July' in period:
                    year_data[year] = population
        
        # Create records
        for year, population in year_data.items():
            records.append(Demographic(
                age=age,
                gender=gender,
                year=int(year),
                population=int(population) if population else 0,
                province=province,
                source_url=f"https://www.ine.es/jaxi/Tabla.htm?tpx=31304",
                fetched_at=datetime.now()
            ))
    
    return records

In [11]:
# Reshape to wide format with Newborns column
def reshape_to_wide(df: pd.DataFrame) -> pd.DataFrame:
    """Convert long format to wide: one row per province-year, with Newborns (0-2) column"""
    df_ages = df[df['age'].notna()].copy()
    
    # Categorize ages
    def categorize_age(age):
        if age <= 2:
            return 'Newborns'
        elif age <= 18:
            return 'Age_3_18'
        elif age <= 30:
            return 'Age_19_30'
        elif age <= 40:
            return 'Age_31_40'
        elif age <= 50:
            return 'Age_41_50'
        elif age <= 60:
            return 'Age_51_60'
        elif age <= 70:
            return 'Age_61_70'
        else:
            return 'Age_71_plus'
    
    df_ages['age_group'] = df_ages['age'].apply(categorize_age)
    
    # Aggregate and pivot
    agg_data = df_ages.groupby(['province', 'year', 'gender', 'age_group'])['population'].sum().reset_index()
    gender_pivot = agg_data.pivot_table(
        index=['province', 'year', 'age_group'],
        columns='gender',
        values='population',
        fill_value=0
    ).reset_index()
    
    wide_data = gender_pivot.pivot_table(
        index=['province', 'year'],
        columns='age_group',
        values=['Total', 'Male', 'Female'],
        fill_value=0
    ).reset_index()
    
    # Flatten columns
    wide_data.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                         for col in wide_data.columns.values]
    
    # Calculate totals
    age_cols = ['Newborns', 'Age_3_18', 'Age_19_30', 'Age_31_40', 
                'Age_41_50', 'Age_51_60', 'Age_61_70', 'Age_71_plus']
    
    wide_data['Total'] = sum(wide_data[f'Total_{col}'] for col in age_cols if f'Total_{col}' in wide_data.columns)
    wide_data['Male'] = sum(wide_data[f'Male_{col}'] for col in age_cols if f'Male_{col}' in wide_data.columns)
    wide_data['Female'] = sum(wide_data[f'Female_{col}'] for col in age_cols if f'Female_{col}' in wide_data.columns)
    
    # Order columns
    base_cols = ['year', 'province']
    gender_cols = ['Total', 'Male', 'Female']
    newborn_cols = [col for col in wide_data.columns if 'Newborns' in col]
    other_age_cols = sorted([col for col in wide_data.columns if '_Age_' in col])
    
    final_cols = base_cols + gender_cols + newborn_cols + other_age_cols
    wide_data = wide_data[[col for col in final_cols if col in wide_data.columns]]
    
    # Sort: National Total first, then alphabetically
    wide_data['sort_key'] = wide_data['province'].apply(lambda x: 0 if x == 'National Total' else 1)
    wide_data = wide_data.sort_values(['sort_key', 'province', 'year']).drop('sort_key', axis=1)
    
    return wide_data

In [12]:
# Export data
def export_demographics(records: list[Demographic], output_dir: str = "../data/processed"):
    """Export to CSV and JSON (wide format only)"""
    import json
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Convert to long format then reshape to wide
    df_long = pd.DataFrame([r.model_dump() for r in records])
    df = reshape_to_wide(df_long)
    
    # Export wide format only
    df.to_csv(f"{output_dir}/ine_demographics.csv", index=False)
    df.to_json(f"{output_dir}/ine_demographics.json", orient='records', indent=2)
    
    # Summary
    latest_year = df['year'].max()
    provinces = df['province'].nunique()
    
    print(f"✅ Exported {len(df):,} rows")
    print(f"📊 Latest year: {latest_year}")
    print(f"📍 Provinces: {provinces}")
    print(f"📁 Output: {output_dir}")
    
    return df

In [14]:
# Fetch and parse provisional data for recent years (2023+)
def fetch_provisional_data():
    """Fetch provisional estimates from Table 56934 (National Total only, 2023-2025)"""
    try:
        raw = fetch_ine_population_by_age_sex(table_id="56934")
        demographics = parse_demographics(raw)
        df_long = pd.DataFrame([r.model_dump() for r in demographics])
        
        # Filter only years newer than 2022 and National Total
        df_long = df_long[(df_long['year'] > 2022) & (df_long['province'] == 'National Total')]
        
        if not df_long.empty:
            df_wide = reshape_to_wide(df_long)
            print(f"✅ Added provisional data for years: {sorted(df_wide['year'].unique())}")
            return df_wide
        else:
            print("⚠️ No provisional data found for 2023+")
            return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ Could not fetch provisional data: {e}")
        return pd.DataFrame()

In [15]:
# Run pipeline
print("🔄 Fetching INE demographics (1971-2022)...")
raw = fetch_ine_population_by_age_sex()
print(f"✅ Received {len(raw):,} series\n")

print("🔍 Parsing data...")
demographics = parse_demographics(raw)
print(f"✅ Parsed {len(demographics):,} records\n")

print("💾 Exporting...")
df = export_demographics(demographics)

print("\n🔄 Fetching provisional data (2023+)...")
df_provisional = fetch_provisional_data()

if not df_provisional.empty:
    # Combine with main data
    df_combined = pd.concat([df, df_provisional], ignore_index=True)
    df_combined = df_combined.sort_values(['province', 'year'])
    
    # Re-export with provisional data
    import json
    df_combined.to_csv("../data/processed/ine_demographics.csv", index=False)
    df_combined.to_json("../data/processed/ine_demographics.json", orient='records', indent=2)
    
    df = df_combined
    print(f"✅ Combined dataset now includes years: {df['year'].min()}-{df['year'].max()}")

print("\n📊 Latest year Spain (sample):")
latest_year = df['year'].max()
sample = df[(df['province'] == 'National Total') & (df['year'] == latest_year)]
print(f"  Year: {latest_year}")
print(f"  Total: {sample['Total'].values[0]:,.0f}")
print(f"  Newborns (0-2): {sample['Total_Newborns'].values[0]:,.0f}")
print(f"  Children (3-18): {sample['Total_Age_3_18'].values[0]:,.0f}")

print("\n🎯 Complete!")

🔄 Fetching INE demographics (1971-2022)...
✅ Received 16,377 series

🔍 Parsing data...
✅ Parsed 770,346 records

💾 Exporting...
✅ Exported 2,808 rows
📊 Latest year: 2022
📍 Provinces: 54
📁 Output: ../data/processed

🔄 Fetching provisional data (2023+)...
✅ Added provisional data for years: [np.int64(2023), np.int64(2024), np.int64(2025)]
✅ Combined dataset now includes years: 1971-2025

📊 Latest year Spain (sample):
  Year: 2025
  Total: 50,678,109
  Newborns (0-2): 991,715
  Children (3-18): 7,510,278

🎯 Complete!
